# NeuralOps Unified Training API

The same `NeuralOpsTrainer` entry point supports standard fine-tuning, LoRA, knowledge distillation, and the hybrid mode.

In [1]:
import copy
import torch
from torch.utils.data import Dataset
from transformers import DistilBertConfig, DistilBertForSequenceClassification, TrainingArguments
from neuralops.trainer import NeuralOpsTrainer

class TinySST2(Dataset):
    def __init__(self, n=4):
        self.input_ids = torch.randint(0, 1000, (n, 16))
        self.attention_mask = torch.ones_like(self.input_ids)
        self.labels = torch.randint(0, 2, (n,))
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i], "attention_mask": self.attention_mask[i], "labels": self.labels[i]}

dataset = TinySST2()
def new_model():
    return DistilBertForSequenceClassification(DistilBertConfig(vocab_size=1000, dim=64, hidden_dim=128, n_layers=1, n_heads=4, num_labels=2))
def args(name):
    return TrainingArguments(output_dir=f"/tmp/neuralops_{name}", per_device_train_batch_size=2, per_device_eval_batch_size=2, max_steps=1, logging_steps=1, eval_strategy="no", report_to=[])


/mnt/d/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
standard = NeuralOpsTrainer(model=new_model(), args=args("standard"), train_dataset=dataset, eval_dataset=dataset)
standard_result = standard.train()
standard_metrics = standard.evaluate()
print("standard", standard_result.global_step, standard_metrics["eval_loss"])

Step,Training Loss
1,0.695398


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 700.45it/s]

Training Loss,Validation Loss,Step
0.695398,0.686354,1


standard 1 0.686353862285614


In [3]:
lora = NeuralOpsTrainer(model=new_model(), args=args("lora"), train_dataset=dataset, eval_dataset=dataset, peft_method="lora", lora_r=2)
lora_result = lora.train()
lora_metrics = lora.evaluate()
print("lora", lora_result.global_step, lora_metrics["eval_loss"])

Step,Training Loss
1,0.704396


Training Loss,Validation Loss,Step
0.704396,0.696070,1


lora 1 0.69607013463974


In [4]:
teacher = new_model()
kd = NeuralOpsTrainer(model=new_model(), args=args("kd"), train_dataset=dataset, eval_dataset=dataset, teacher_model=teacher)
kd_result = kd.train()
kd_metrics = kd.evaluate()
print("kd", kd_result.global_step, kd_metrics["eval_loss"])

Step,Training Loss
1,0.345287


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 728.56it/s]

Training Loss,Validation Loss,Step
0.345287,0.340473,1


kd 1 0.3404729962348938


In [5]:
hybrid = NeuralOpsTrainer(model=new_model(), args=args("hybrid"), train_dataset=dataset, eval_dataset=dataset, peft_method="lora", lora_r=2, teacher_model=new_model())
hybrid_result = hybrid.train()
hybrid_metrics = hybrid.evaluate()
print("hybrid", hybrid_result.global_step, hybrid_metrics["eval_loss"])

Step,Training Loss
1,0.352221


Training Loss,Validation Loss,Step
0.352221,0.348120,1


hybrid 1 0.3481196463108063
